# Pretrain Commutative CNN Encoder

Load the shared unlabeled pretraining dataset and save commutative CNN encoder weights for downstream classification notebooks.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from sklearn.model_selection import train_test_split

from src.ml import (
    CommutativeCNNClassifier,
    CommutativeCNNConfig,
    CommutativeCNNPretrainingConfig,
    LossWeightConfig,
    OptimizationConfig,
    augment_training_tensors_with_rotations,
    create_experiment_run,
    load_commutative_cnn_pretraining_config,
    persist_pretraining_artifacts,
    write_commutative_cnn_pretraining_config,
)
from src.tensor_utils import load_unlabeled_tensor_dataset


In [2]:
# User inputs

unlabeled_dataset_path = Path(".dataset_cache/unlabeled_active_high_mid_low_t20_z5_y96_x96_chunks")
experiment_output_dir = Path("artifacts/pretrained_commutative_cnn")
experiment_run = create_experiment_run(experiment_output_dir, "10C_pretrain_commutative_cnn")
pretrained_encoder_path = Path(experiment_run.run_dir) / f"{experiment_run.experiment_id}_encoder_state.pt"
validation_fraction = 0.15
train_num_random_rotations = 0
rotation_range_degrees = 0.0

model_config = CommutativeCNNConfig(
    spatial_conv_channels=(16, 32),
    spatial_kernel_size_z=(3, 1),
    spatial_kernel_size_xy=(5, 3),
    spatial_stride_z=(1, 1),
    spatial_stride_xy=(1, 1),
    spatial_pool_kernel_z=(1, 1),
    spatial_pool_kernel_xy=(2, 2),
    spatial_pool_stride_z=(1, 1),
    spatial_pool_stride_xy=(2, 2),
    temporal_st_channels=(48, 64),
    temporal_st_kernel_sizes=(5, 3),
    temporal_ts_channels=(32, 48, 64),
    temporal_ts_kernel_sizes=(7, 5, 3),
    spatial_agg_channels=(32, 64),
    spatial_agg_kernel_size_z=(3, 1),
    spatial_agg_kernel_size_xy=(3, 3),
    spatial_agg_stride_z=(1, 1),
    spatial_agg_stride_xy=(1, 1),
    spatial_agg_pool_kernel_z=(1, 1),
    spatial_agg_pool_kernel_xy=(1, 2),
    spatial_agg_pool_stride_z=(1, 1),
    spatial_agg_pool_stride_xy=(1, 2),
    patch_size_z=1,
    patch_size_xy=16,
    embedding_dim=64,
    num_prototypes=64,
    probe_region_grid=(1, 2, 2),
    probe_time_bins=8,
    probe_frequency_bins=4,
    dropout=0.25,
    normalization="group",
)
optimization_config = OptimizationConfig(
    batch_size=8,
    epochs=70,
    learning_rate=3e-5,
    weight_decay=1e-3,
    early_stopping_patience=12,
    early_stopping_min_delta=5e-5,
    early_stopping_start_epoch=8,
    early_stopping_monitor="self_probe_loss",
    early_stopping_smoothing="median",
    early_stopping_smoothing_window=3,
    training_plot_dir=str(Path(experiment_run.loss_plot_dir) / "pretraining"),
    training_plot_every_n_epochs=2,
    training_plot_smoothing_window=5,
    scheduler_patience=3,
    scheduler_factor=0.5,
    scheduler_min_lr=1e-6,
    validation_split=0.0,
    random_state=0,
    standardize=True,
    device=None,
    verbose=True,
)
loss_weight_config = LossWeightConfig(
    lambda_cross=0.0,
    cross_warmup_epochs=0,
    cross_ramp_epochs=0,
    prototype_temperature=0.25,
    prototype_alignment_weight=0.0,
    prototype_warmup_epochs=16,
    prototype_ramp_epochs=36,
    latent_alignment_weight=0.0,
    lambda_align=0.0,
    probe_mask_probability=1.0,
    probe_alpha_local=1.0,
    probe_alpha_region_time=1.0,
    probe_alpha_derivative=0.25,
    probe_alpha_frequency=0.10,
    probe_alpha_correlation=0.05,
)

pretraining_config = CommutativeCNNPretrainingConfig(
    unlabeled_dataset_path=unlabeled_dataset_path,
    pretrained_encoder_path=pretrained_encoder_path,
    validation_fraction=validation_fraction,
    train_num_random_rotations=train_num_random_rotations,
    rotation_range_degrees=rotation_range_degrees,
    model_config=model_config,
    optimization_config=optimization_config,
    loss_weight_config=loss_weight_config,
)
pretraining_config_path = write_commutative_cnn_pretraining_config(pretraining_config)
pretraining_config = load_commutative_cnn_pretraining_config(pretraining_config_path)
print(f"Experiment id: {experiment_run.experiment_id}")
print(f"Experiment run folder: {Path(experiment_run.run_dir).resolve()}")
print(f"Commutative CNN pretraining config in {pretraining_config_path}")
print(f"Pretraining loss PDFs: {Path(optimization_config.training_plot_dir).resolve()}")
print(f"Pretrained encoder checkpoint target: {pretrained_encoder_path.resolve()}")
pretraining_config


Experiment id: 10C_pretrain_commutative_cnn_20260623_093629
Experiment run folder: /run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/ZebraFish/artifacts/pretrained_commutative_cnn/runs/10C_pretrain_commutative_cnn_20260623_093629
Commutative CNN pretraining config in /run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/ZebraFish/artifacts/pretrained_commutative_cnn/config.yaml
Pretraining loss PDFs: /run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/ZebraFish/artifacts/pretrained_commutative_cnn/runs/10C_pretrain_commutative_cnn_20260623_093629/loss_plots/pretraining
Pretrained encoder checkpoint target: /run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/ZebraFish/artifacts/pretrained_commutative_cnn/runs/10C_pretrain_commutative_cnn_20260623_093629/10C_pretrain_commutative_cnn_20260623_093629_encoder_state.pt


CommutativeCNNPretrainingConfig(unlabeled_dataset_path=PosixPath('.dataset_cache/unlabeled_active_high_mid_low_t20_z5_y96_x96_chunks'), pretrained_encoder_path=PosixPath('artifacts/pretrained_commutative_cnn/runs/10C_pretrain_commutative_cnn_20260623_093629/10C_pretrain_commutative_cnn_20260623_093629_encoder_state.pt'), validation_fraction=0.15, train_num_random_rotations=0, rotation_range_degrees=0.0, model_config=CommutativeCNNConfig(spatial_conv_channels=(16, 32), spatial_kernel_size_z=(3, 1), spatial_kernel_size_xy=(5, 3), spatial_stride_z=(1, 1), spatial_stride_xy=(1, 1), spatial_pool_kernel_z=(1, 1), spatial_pool_kernel_xy=(2, 2), spatial_pool_stride_z=(1, 1), spatial_pool_stride_xy=(2, 2), temporal_st_channels=(48, 64), temporal_st_kernel_sizes=(5, 3), temporal_ts_channels=(32, 48, 64), temporal_ts_kernel_sizes=(7, 5, 3), spatial_agg_channels=(32, 64), spatial_agg_kernel_size_z=(3, 1), spatial_agg_kernel_size_xy=(3, 3), spatial_agg_stride_z=(1, 1), spatial_agg_stride_xy=(1, 1),

In [ ]:
unlabeled_dataset = load_unlabeled_tensor_dataset(unlabeled_dataset_path)
train_indices, val_indices = train_test_split(
    range(len(unlabeled_dataset["tensors"])),
    test_size=validation_fraction,
    random_state=optimization_config.random_state,
    shuffle=True,
)
X_train_base = unlabeled_dataset["tensors"][train_indices]
X_val = unlabeled_dataset["tensors"][val_indices]
metadata_train_base = unlabeled_dataset["metadata"].iloc[train_indices].reset_index(drop=True)
metadata_val = unlabeled_dataset["metadata"].iloc[val_indices].reset_index(drop=True)
X_train, _, metadata_train = augment_training_tensors_with_rotations(
    X_train_base,
    [0] * len(X_train_base),
    metadata=metadata_train_base,
    num_random_rotations=train_num_random_rotations,
    rotation_range_degrees=rotation_range_degrees,
    random_state=optimization_config.random_state,
)
{
    "all_tensors": unlabeled_dataset["tensors"].shape,
    "all_metadata": unlabeled_dataset["metadata"].shape,
    "train_base_tensors": X_train_base.shape,
    "train_tensors": X_train.shape,
    "val_tensors": X_val.shape,
    "train_base_metadata": metadata_train_base.shape,
    "train_metadata": metadata_train.shape,
    "val_metadata": metadata_val.shape,
}


## Output Review

The previous 10C/v9 run reached a usable checkpoint but validation self-probe loss became volatile soon after the best epoch, while training self-probe continued to improve. That pattern is consistent with over-coupled auxiliary pressure and small-batch BatchNorm instability rather than a simple optimization failure.

This next diagnostic run tests stability first: the commutative CNN uses `normalization="group"`, `lambda_cross=0.0`, no prototype/latent alignment, `learning_rate=3e-5`, `weight_decay=1e-3`, and validation self-probe monitoring starts at epoch `8`. If validation self-probe remains stable, cross-probe and prototype objectives can be reintroduced in later runs one at a time.

In [ ]:
%%time
model = CommutativeCNNClassifier(
    model_config=model_config,
    optimization_config=optimization_config,
    loss_weight_config=loss_weight_config,
)
model.pretrain(X_train, validation_data=X_val)
pretrained_encoder_path = model.save_pretrained_encoder(pretrained_encoder_path)
pretraining_artifacts = persist_pretraining_artifacts(
    output_dir=experiment_output_dir,
    estimator=model,
    config=pretraining_config,
    experiment_prefix="10C_pretrain_commutative_cnn",
    experiment_id=experiment_run.experiment_id,
    pretrained_encoder_path=pretrained_encoder_path,
    loss_plot_dirs=[optimization_config.training_plot_dir],
    analysis=(
        "Planned diagnostic run: test whether GroupNorm plus removal of cross/prototype/latent auxiliary "
        "pressure stabilizes validation self-probe loss after the prior v9 run diverged after its best epoch. "
        "After the run, replace this with best epoch, train/validation self-probe behavior, and checkpoint quality."
    ),
    next_round_proposal=(
        "If validation self-probe is stable, run 13C from this encoder and then reintroduce cross-probe "
        "pressure at a small weight. If validation remains volatile, keep cross/prototype disabled and test "
        "InstanceNorm or a smaller learning rate."
    ),
)
pretraining_artifacts


In [ ]:
model.pretrain_history_.tail()